In [1]:
# Packages 
import polars as pl

In [2]:
# Time constants
seconds_in_day = 60 * 60 * 24 
minutes_per_week = 7 * 24 * 60 
n_weeks = 8     
eight_weeks_seconds = n_weeks * minutes_per_week * 60

In [3]:
# Load data and filter for human users & first 8 weeks of data
df = (
    pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz", has_header=False, separator=",",
                new_columns=['time','src_user','dest_user','src_comp','dest_comp',
                              'auth_type','logon_type','auth_orientation','outcome'])
    .filter(pl.col('src_user').str.starts_with('U'))
    .filter(pl.col('time') < eight_weeks_seconds)
    .collect(engine='streaming')
)

In [4]:
df.show(5)

time,src_user,dest_user,src_comp,dest_comp,auth_type,logon_type,auth_orientation,outcome
i64,str,str,str,str,str,str,str,str
1,"""U101@DOM1""","""C1862$@DOM1""","""C1862""","""C1862""","""?""","""?""","""AuthMap""","""Success"""
1,"""U101@DOM1""","""U101@DOM1""","""C1862""","""C1862""","""Negotiate""","""Interactive""","""LogOn""","""Success"""
1,"""U10@DOM1""","""U10@DOM1""","""C229""","""C229""","""Kerberos""","""Network""","""LogOn""","""Success"""
1,"""U10@DOM1""","""U10@DOM1""","""C62""","""C528""","""Kerberos""","""Network""","""LogOn""","""Success"""
1,"""U1137@DOM1""","""U1137@DOM1""","""C1065""","""C1065""","""?""","""Network""","""LogOff""","""Success"""


In [6]:
# Number of human auth events
df.select(pl.len()).item()

326819512

In [7]:
# Number of unique source users
df.select(pl.col("src_user").n_unique()).item()

30053